# Pharmacophore + PyMOL script from a single trajectory

Companion notebook for *Dynamic Pharmacophore Mapping for Intrinsically Disordered Drug Targets* ([paper](#) · [SI](#)).

Upload one protein-ligand trajectory + topology -- either a full simulation or a single subset produced by **`01_split_trajectory_pca_graph.ipynb`** -- and this notebook runs the full pharmacophore pipeline on it:
1. Ligand ring / H-bond donor detection (RDKit)
2. Aromatic, H-bond, and general contact probabilities
3. Ligand-frame alignment and the negative-space voxel grid (growth space vs. pharmacophore)
4. Pharmacophore occupancy maps and growth-space chemical feature maps
5. `.mrc` volume export + a `.pml` PyMOL script that loads all of it

Mirrors the per-subset loop body of `pharmacophores_analysis.ipynb` from the pipeline's development repo, applied here to one uploaded trajectory instead of a batch list -- no new algorithms.

**Tip:** `Runtime -> Change runtime type -> GPU` speeds up the voxel-grid steps (`compute_negative_space*`, `compute_pharmacophore_maps*`, `compute_growth_space_features*` all auto-detect CUDA via PyTorch and fall back to multithreaded CPU otherwise). Not required, just faster.

## 1. Setup

In [ ]:
%%capture
%pip install -q mdtraj MDAnalysis rdkit mrcfile deeptime hdbscan pyblock networkx

In [ ]:
import os, urllib.request

MODULE_URL = "https://raw.githubusercontent.com/emalacs/Scalone_IDP_Pharmacophore_Mapping_2026/main/pharmacophore_utils.py"
if not os.path.exists("pharmacophore_utils.py"):
    urllib.request.urlretrieve(MODULE_URL, "pharmacophore_utils.py")

import pharmacophore_utils as pu

## 2. Upload topology + trajectory

In [ ]:
from google.colab import files

print('Select the topology file (.gro/.pdb/...)')
topology_upload = files.upload()
TOPOLOGY = next(iter(topology_upload))

print('Select the trajectory file (.xtc/.dcd/...)')
trajectory_upload = files.upload()
TRAJECTORY = next(iter(trajectory_upload))

print(f'Topology  : {TOPOLOGY}')
print(f'Trajectory: {TRAJECTORY}')

In [ ]:
# Alternative for large trajectories: mount Google Drive and point directly at the files instead
# of using the upload widget above.
#
# from google.colab import drive
# drive.mount('/content/drive')
# TOPOLOGY   = '/content/drive/MyDrive/path/to/system.gro'
# TRAJECTORY = '/content/drive/MyDrive/path/to/traj.xtc'

## 3. Settings

In [ ]:
SIM_NAME       = 'my_subset'   # used as the output folder name
LIGAND_RESNAME = None          # None -> auto-detect from topology
STRIDE         = 1
OFFSET         = 0
OUTPUT_DIR     = f'./output/{SIM_NAME}'

# -- Voxel grid settings -------------------------------------------------------
# dx=0.5 A: fast, default. dx=0.3 A: finer grid (used in production). dx=0.15 A: max detail.
DX             = 0.3   # voxel edge length (A)
GAUSSIAN_SIGMA = 1.0   # smoothing sigma passed to save_mrc_files() -- None = no smoothing

SHELL_INNER_RADIUS  = 1.0   # A -- inner boundary of the analysis shell (beyond ligand vdW surface)
SHELL_OUTER_RADIUS  = 5.0   # A -- outer boundary of the analysis shell
PROTEIN_VDW_RADIUS  = 1.5   # A -- protein atom exclusion radius
MIN_FREE_FRACTION   = 0.7   # threshold: free >= 70% of the trajectory -> growth space

AROMATIC_CUTOFF    = 4.5   # A -- growth-space aromatic feature cutoff
HBOND_CUTOFF       = 3.5   # A -- growth-space H-bond feature cutoff
HYDROPHOBIC_CUTOFF = 5.0   # A -- growth-space hydrophobic feature cutoff

SCORE_PERCENTILE_TIERS = [1, 5, 10, 20]   # thinned top-N% map variants written alongside the full maps

## 4. Load trajectory

In [ ]:
sim = pu.PharmacophoreTrajectory(TOPOLOGY, TRAJECTORY)
sim.load(ligand_resname=LIGAND_RESNAME, stride=STRIDE, offset=OFFSET)
import os; os.makedirs(f'{OUTPUT_DIR}/plots', exist_ok=True)

## 5. Ligand chemical features
Ring detection is required for aromatic contacts below. The labeled 2D structure helps pick atom names for the alignment selection in the next section.

In [ ]:
result = pu.draw_molecule_with_labels(sim.ligand_mol)
if result:
    img, mol = result
    display(img)

In [ ]:
rings = sim.get_ligand_rings()
print(f'{len(rings)} ring(s) detected')
for r in rings:
    print(r)

# Override if auto-detection is wrong (use atom names from the labeled structure above):
# sim._manual_rings = [
#     {'atom_names': ['C5', 'C9', 'C13', 'C10', 'C6', 'C1'], 'aromatic': True},
# ]

## 6. Contact probabilities
Aromatic and H-bond contacts are required inputs for the contact/non-contact split in the pharmacophore maps (Section 8) -- not just diagnostics.

In [ ]:
probs = sim.compute_aromatic_contacts()
print(probs[probs['aromatic_stacking'] > 0][['aromatic_stacking', 'aromatic_pstacking', 'aromatic_tstacking']])

In [ ]:
fig_aro = sim.plot_contact_probability(df=probs, data='aromatic_stacking', add_error_bars=False)
fig_aro.show()
sim.save_plot(fig_aro, f'{OUTPUT_DIR}/plots/{SIM_NAME}_aromatic_contacts.html')

In [ ]:
# General distance-based contact probability -- bound fraction / Kd, not required downstream but
# a useful sanity check that the ligand actually engages the protein in this trajectory.
contact_probs = sim.compute_contact_probability(cutoff=0.6, scheme='closest-heavy')
print(contact_probs[contact_probs['contact_probability'] > 0][['contact_probability', 'Kd']])
if sim.bound_fraction is not None:
    print(f'\nBound fraction: {sim.bound_fraction[0]:.3f} +/- {sim.bound_fraction[1]:.3f}')
    print(f'Kd: {sim.kd[0]:.3f} +/- {sim.kd[1]:.3f} mM')

In [ ]:
hbond_pairs = sim.get_ligand_hbond_pairs()
print(f'H-bond donor pairs: {hbond_pairs}')

hbond_probs = sim.compute_hbond_contacts()
print(hbond_probs[hbond_probs['Hbonds_average'] > 0][['Hbonds_average', 'Hbonds_PD_average', 'Hbonds_LD_average']])

## 7. Align + negative-space voxel grid
The voxel grid is built from raw coordinates, so all frames must share the same ligand reference frame. Set `ALIGN_SELECTION` using atom names from the labeled structure in Section 5 -- three or four atoms spanning the ligand's rigid core work well. If left as `None`, the grid uses raw (unaligned) coordinates, which is only safe for an already-rigid-body-aligned trajectory.

In [ ]:
ALIGN_SELECTION = None   # e.g. "resname LIG and (name C5 or name C6 or name C9 or name C17)"

if ALIGN_SELECTION:
    sim.ligand_align(ALIGN_SELECTION)
else:
    print("Note: ALIGN_SELECTION not set -- using raw trajectory coordinates for the voxel grid.")

In [ ]:
neg_space = sim.compute_negative_space_threaded(
    shell_inner_radius=SHELL_INNER_RADIUS,
    shell_outer_radius=SHELL_OUTER_RADIUS,
    dx=DX,
    protein_vdw_radius=PROTEIN_VDW_RADIUS,
    min_free_fraction=MIN_FREE_FRACTION,
)
print('\nVolumes (A^3):')
for k, v in neg_space['volumes'].items():
    print(f'  {k:20s}: {v:.1f}')

## 8. Pharmacophore + growth-space feature maps

In [ ]:
maps_contested, maps_full = sim.compute_pharmacophore_maps_threaded()
for label, maps in [('Pharmacophore', maps_contested), ('Full shell', maps_full)]:
    print(f'\n{label}:')
    for key in ['aromatic_occupancy', 'hydrophobic_occupancy',
                'hbond_donors_occupancy', 'hbond_acceptors_occupancy',
                'positive_occupancy', 'negative_occupancy']:
        arr = maps.get(key)
        nz = arr[arr > 0] if arr is not None else []
        if len(nz):
            print(f'  {key:<26}: {len(nz):>6} voxels  max={nz.max():.3f}  mean={nz.mean():.3f}')
        else:
            print(f'  {key:<26}: no voxels above zero')

In [ ]:
growth_features = sim.compute_growth_space_features_threaded(
    aromatic_cutoff=AROMATIC_CUTOFF,
    hbond_cutoff=HBOND_CUTOFF,
    hydrophobic_cutoff=HYDROPHOBIC_CUTOFF,
)
for name, key in [
    ('Aromatic', 'aromatic_score'), ('Hydrophobic', 'hydrophobic_score'),
    ('HBond Donor', 'hbond_donor_score'), ('HBond Acceptor', 'hbond_acceptor_score'),
    ('Positive Charge', 'positive_charge_score'), ('Negative Charge', 'negative_charge_score'),
]:
    arr = growth_features[key]
    nz = arr[arr > 0]
    print(f'{name:<18}: {len(nz):>6} voxels  max={nz.max():.3f}' if len(nz) else f'{name:<18}: no voxels')

## 9. Save volumes + dot clouds
`.mrc` volumes for PyMOL's isomesh/volume rendering, plus per-frame atom-position "dot cloud" PDBs (aromatic ring centroids, H-bond donor/acceptor atoms, hydrophobic centroids) split by satisfied / unsatisfied ligand contact.

In [ ]:
sim.save_mrc_files(OUTPUT_DIR, gaussian_sigma=GAUSSIAN_SIGMA, score_percentile_tiers=SCORE_PERCENTILE_TIERS)

In [ ]:
sim.compute_residue_dot_positions(mask_key='full_shell_mask', stride=STRIDE)
sim.save_dots_pdb(OUTPUT_DIR)
sim.save_aromatic_ring_trajectory(OUTPUT_DIR, stride=1)

## 10. Satisfied/unsatisfied diff maps
Cancels rotational/rotameric noise shared between the `_contacts`/`_non_contacts` occupancy pair written in Section 9. A residue that is genuinely docked (e.g. a stacked aromatic ring) can still librate/flip locally -- its centroid barely moves, but individual frames can cross the contact-geometry cutoff and get classified as unsatisfied that frame, inflating the `_non_contacts` map at voxels that are really part of the stable satisfied site.

For each category (`aromatic_occupancy`, `hbond_donors_occupancy`, `hbond_acceptors_occupancy`) and map-set (`pharmacophore_full`, `pharmacophore_contested`) written above, this writes `{category}_diff.mrc` (`diff = non_contact - contact`, signed, range `[-1, 1]`) and `{category}_diff_ratio.mrc` (`ratio = diff / (contact + non_contact)`, per-voxel normalized so a category-wide base-rate imbalance -- e.g. H-bond's much broader unsatisfied atom pool -- doesn't dominate the raw diff), plus sign-split, percentile-tiered variants of `ratio` for isomeshing each side independently in PyMOL. Aromatic is the most trustworthy category for this technique; treat H-bond diff/ratio maps with caution (see `DEVELOPMENT.md` in the pipeline's development repo for why).

In [ ]:
diff_results = {}
for map_set in pu.DIFF_MAP_SETS:
    map_dir = os.path.join(OUTPUT_DIR, map_set)
    if not os.path.isdir(map_dir):
        continue
    print(f'\n{map_set}/ …')
    diff_results[map_set] = {}
    for category in pu.DIFF_CATEGORIES:
        diff_results[map_set][category] = pu.diff_category(
            map_dir, category, gaussian_sigma=GAUSSIAN_SIGMA,
            score_percentile_tiers=SCORE_PERCENTILE_TIERS,
        )

In [ ]:
for map_set, categories in diff_results.items():
    for category, result in categories.items():
        if result is None:
            continue
        fig_diff = pu.plot_diff_histogram(
            result['diff'], result['signal_mask'], category,
            title=f'{map_set} / {category}')
        pu.show_figure(fig_diff)

        fig_ratio = pu.plot_diff_histogram(
            result['ratio'], result['signal_mask'], category,
            title=f'{map_set} / {category} (ratio)',
            xlabel='ratio = (non_contact − contact) / (non_contact + contact)')
        pu.show_figure(fig_ratio)

        fig_scatter = pu.plot_diff_magnitude_scatter(
            result['diff'], result['contact'], result['non_contact'],
            result['signal_mask'], category,
            title=f'{map_set} / {category} vs magnitude')
        pu.show_figure(fig_scatter)

## 11. PyMOL script
Writes `view_pharmacophore.pml`, loading the ligand, negative-space volumes, and pharmacophore isomeshes. `load_trajectory=True` also prepends the raw topology/trajectory so the script can play back the ensemble alongside the maps. Paste PyMOL's `get_view` output into `SET_VIEW` afterwards to bake in a fixed camera angle (optional -- a commented template is written when left as `None`).

In [ ]:
SET_VIEW = None   # r"""set_view (\\ ... )""" pasted from PyMOL's `get_view` command

pml_path = sim.write_pymol_script(
    OUTPUT_DIR, score_percentile_tiers=SCORE_PERCENTILE_TIERS,
    set_view=SET_VIEW, load_trajectory=True,
)
print(f'Open with: pymol {pml_path}')

## 12. Download
Zips `OUTPUT_DIR` (MRC volumes, dot-cloud PDBs, `.pml` script, plots) and downloads it. To add this subset to the web viewer, drop the unzipped folder into `output/<subset_name>/` in this repo and list it in `web/src/data/catalog.js`.

In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(SIM_NAME, 'zip', OUTPUT_DIR)
files.download(archive_path)